<a href="https://colab.research.google.com/github/EgzonnOsmanaj/MesoAI/blob/main/EU_AI_ACT_RAG_WBS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG System for EU AI Act — Gen AI & AI Applications Assignment

**Student:** [Your Name]  
**Module:** Gen AI and AI Applications  
**Date:** June 2025

---

## 1. Domain, Data Sourcing & Justification

### Domain
This project builds a Retrieval-Augmented Generation (RAG) system over the **EU AI Act** (Regulation (EU) 2024/1689), the landmark European legislation that entered into force on 1 August 2024. The full text runs to ~144 pages of dense legal language organised into recitals, articles, annexes and definitions.

### Data Source
The official PDF is downloaded directly from EUR-Lex, the EU's public legal database:  
`https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689`  
This is a publicly available, authoritative source with no licensing restrictions on use.

### Why RAG Rather Than Plain Prompting?

1. **Recency.** The EU AI Act was published in July 2024. Most LLM training corpora have a knowledge cutoff before the final text was adopted; the model may have seen drafts but not the enacted regulation with correct article numbers, annex references and application dates.
2. **Precision.** Legal answers require exact citation of articles, recitals and annexes. A model generating from parametric memory is prone to hallucinating article numbers or conflating provisions from earlier legislative drafts.
3. **Length & specificity.** The Act defines ~60 technical terms, lists 8 high-risk application areas, 7 prohibited practices, 2 GPAI tiers, and a tiered application timeline. No practical prompt can surface all relevant passages for an arbitrary legal question without retrieval.
4. **Verifiability.** Regulators and compliance officers need page-level citations. RAG grounds every answer in retrieved passages, making outputs auditable in a way that pure generation cannot be.

RAG is therefore the correct architectural choice: it combines the language understanding of an LLM with faithful grounding in the authoritative text.


## 2. Setup & Imports

In [ ]:
!pip -q install pandas numpy scikit-learn sentence-transformers faiss-cpu rank-bm25 pymupdf requests beautifulsoup4 lxml google-genai tqdm

In [ ]:
import os
import re
import json
import time
import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dataclasses import dataclass

import fitz
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai

In [ ]:
DATA_DIR = "data"
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
EVAL_DIR = os.path.join(DATA_DIR, "eval")
for d in [DATA_DIR, RAW_DIR, PROCESSED_DIR, EVAL_DIR]:
    os.makedirs(d, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "eu_ai_act.pdf") # Added this line
PDF_URL = "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689"

if not os.path.exists(PDF_PATH):
    print(f"Downloading PDF from {PDF_URL} to {PDF_PATH}")
    response = requests.get(PDF_URL)
    response.raise_for_status() # Raise an exception for bad status codes
    with open(PDF_PATH, "wb") as f:
        f.write(response.content)
    print("Download complete.")

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
GEMINI_MODEL = "gemini-2.5-flash"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

TOP_K_BM25 = 20
TOP_K_DENSE = 20
TOP_K_RERANK = 5

In [ ]:
import os
from google.colab import userdata

# Fetch the Gemini API key from the Colab secrets manager
GEMINI_API_KEY = userdata.get('GoogleAIAPI')

if not GEMINI_API_KEY:
    raise ValueError("Please set GEMINI_API_KEY in your environment or Colab secrets (named 'GoogleAIAPI') first.")

client = genai.Client(api_key=GEMINI_API_KEY)

## 3. Preprocessing: Chunking & Embedding

### Strategy
The EU AI Act is extracted page-by-page using PyMuPDF. Pages that exceed 1 800 characters are split with a 200-character overlap to avoid cutting mid-sentence while keeping each chunk within the embedding model's effective range. This is a **fixed-size character-level chunking** strategy — simple and robust for a single, well-structured legal document where article boundaries generally fall within single pages.

Each chunk carries metadata: `doc_id`, `page`, `part` (sub-page index). This metadata is propagated through retrieval and surfaced in generated answers as page citations.


In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    text: str
    metadata: dict

def clean_text(text):
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def split_long_text(text, max_chars=1800, overlap=200):
    text = clean_text(text)
    if len(text) <= max_chars:
        return [text]
    parts = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        parts.append(text[start:end].strip())
        if end == len(text):
            break
        start = end - overlap
    return [p for p in parts if p]

In [ ]:
def extract_pdf_pages(pdf_path):
    doc = fitz.open(pdf_path)
    rows = []
    for i in range(len(doc)):
        page = doc.load_page(i)
        text = clean_text(page.get_text("text"))
        if text:
            rows.append({
                "doc_id": "eu_ai_act",
                "page": i + 1,
                "title": "EU AI Act",
                "text": text
            })
    return pd.DataFrame(rows)


pages_df = extract_pdf_pages(PDF_PATH)
pages_df.head()

In [ ]:
def build_chunks_from_pdf_pages(pages_df):
    chunks = []
    for _, row in pages_df.iterrows():
        parts = split_long_text(row["text"], max_chars=1800, overlap=200)
        for j, part in enumerate(parts):
            chunks.append(
                Chunk(
                    chunk_id=f"{row['doc_id']}_page_{int(row['page'])}_{j}",
                    doc_id=row["doc_id"],
                    text=part,
                    metadata={
                        "title": row["title"],
                        "page": int(row["page"]),
                        "part": j
                    }
                )
            )
    return chunks

chunks = build_chunks_from_pdf_pages(pages_df)
len(chunks), chunks[0]

In [ ]:
def chunks_to_dataframe(chunks):
    rows = []
    for c in chunks:
        row = {
            "chunk_id": c.chunk_id,
            "doc_id": c.doc_id,
            "text": c.text
        }
        row.update(c.metadata)
        rows.append(row)
    return pd.DataFrame(rows)

chunks_df = chunks_to_dataframe(chunks)
chunks_df.to_csv(os.path.join(PROCESSED_DIR, "chunks.csv"), index=False)
chunks_df.head()

### Embedding & Vector Store

Chunks are encoded with `all-MiniLM-L6-v2` (384-dim, normalised) and stored in a FAISS `IndexFlatIP` (inner product = cosine similarity on unit vectors). A BM25 index over the same corpus is built in parallel for the hybrid retrieval stage.


In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

chunk_texts = [c.text for c in chunks]
chunk_embeddings = embed_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dim = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(chunk_embeddings)

print("Chunks:", len(chunks))
print("Embedding dim:", dim)
print("FAISS size:", faiss_index.ntotal)

In [ ]:
tokenized_corpus = [c.text.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
reranker = CrossEncoder(RERANK_MODEL_NAME)

## 4. Advanced RAG Pipeline

### Advanced Improvements Implemented

| Improvement | Description |
|---|---|
| **Hybrid retrieval (BM25 + Dense)** | BM25 captures exact legal term matches ("high-risk", "systemic risk"); dense retrieval captures semantic similarity. Both are required for legal text. |
| **Reciprocal Rank Fusion (RRF)** | Fuses the two ranked lists without requiring score normalisation, assigning higher combined rank to candidates appearing highly in both lists. |
| **Cross-encoder reranking** | A `ms-marco-MiniLM-L-6-v2` cross-encoder jointly re-scores the query–chunk pair, producing much more accurate relevance scores than bi-encoder cosine similarity alone. |
| **Rule-based query rewriting** | Normalises informal legal shorthand ("AI Act" → "Regulation (EU) 2024/1689"; "high risk" → "high-risk AI system") to match the Act's own terminology before retrieval. |

### Baseline vs. Enhanced

| Stage | Baseline | Enhanced |
|---|---|---|
| Query | Raw user query | Rewritten query |
| Retrieval | Dense (FAISS) only | BM25 + Dense + RRF |
| Re-ranking | None | Cross-encoder (top-20 → top-5) |
| Generation | Gemini + context | Gemini + context |


In [ ]:
def dense_retrieve(query, top_k=10):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = faiss_index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        c = chunks[idx]
        results.append({"chunk": c, "score": float(score), "method": "dense"})
    return results

def bm25_retrieve(query, top_k=10):
    scores = bm25.get_scores(query.lower().split())
    idxs = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in idxs:
        c = chunks[idx]
        results.append({"chunk": c, "score": float(scores[idx]), "method": "bm25"})
    return results

def reciprocal_rank_fusion(result_lists, k=60):
    fused = {}
    chunk_map = {}
    for res_list in result_lists:
        for rank, item in enumerate(res_list, start=1):
            cid = item["chunk"].chunk_id
            fused[cid] = fused.get(cid, 0.0) + 1.0 / (k + rank)
            chunk_map[cid] = item["chunk"]
    merged = [{"chunk": chunk_map[cid], "score": score} for cid, score in fused.items()]
    merged.sort(key=lambda x: x["score"], reverse=True)
    return merged

def rerank(query, candidates, top_k=5):
    pairs = [(query, c["chunk"].text) for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [{"chunk": item[0]["chunk"], "score": float(item[1])} for item in ranked[:top_k]]

In [ ]:
def rewrite_query(query):
    q = query.strip()
    q = re.sub(r"\bAI Act\b", "Regulation (EU) 2024/1689", q, flags=re.I)
    q = re.sub(r"\bhigh risk\b", "high-risk AI system", q, flags=re.I)
    return q

## 5. Generation with Retrieved Context

### Prompt Engineering

The prompt instructs the model to:
- Act as an expert EU legal assistant
- Answer **only** from the provided context (grounding constraint)
- Explicitly say "I do not have enough evidence" if the context is insufficient (reduces hallucination)
- Cite page numbers in the answer (verifiability)

This is a strict "closed-book" prompt pattern appropriate for compliance use cases where hallucinated legal claims are unacceptable.


In [ ]:
def generate_with_gemini(prompt, model=GEMINI_MODEL):
    response = client.models.generate_content(
        model=model,
        contents=prompt
    )
    return response.text

In [ ]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join([
        f"[{i+1}] Page {c.metadata.get('page','?')}:\n{c.text}"
        for i, c in enumerate(retrieved_chunks)
    ])
    return f"""
You are an expert EU legal assistant.
Answer only from the provided context.
If the answer is not supported, say you do not have enough evidence.
Cite the page numbers where relevant.

Question:
{query}

Context:
{context}

Answer:
""".strip()

In [ ]:
def baseline_answer(query, top_k=5):
    dense_hits = dense_retrieve(query, top_k=TOP_K_DENSE)
    top_chunks = [x["chunk"] for x in dense_hits[:top_k]]
    prompt = build_prompt(query, top_chunks)
    answer = generate_with_gemini(prompt)
    return {
        "query": query,
        "rewritten_query": query,
        "retrieved": top_chunks,
        "answer": answer
    }

In [ ]:
def enhanced_answer(query, top_k=5):
    q = rewrite_query(query)

    bm25_hits = bm25_retrieve(q, top_k=TOP_K_BM25)
    dense_hits = dense_retrieve(q, top_k=TOP_K_DENSE)

    fused = reciprocal_rank_fusion([bm25_hits, dense_hits])
    top_for_rerank = fused[:20]

    reranked = rerank(q, top_for_rerank, top_k=top_k)
    top_chunks = [x["chunk"] for x in reranked]

    prompt = build_prompt(query, top_chunks)
    answer = generate_with_gemini(prompt)

    return {
        "query": query,
        "rewritten_query": q,
        "retrieved": top_chunks,
        "answer": answer
    }

### Quick Smoke Test (3 sample queries)

In [ ]:
test_queries = [
    "What is the purpose of the AI Act?",
    "When do the transparency rules start to apply?",
    "What are high-risk AI systems?",
    "Which AI practices are prohibited?",
    "What obligations apply to providers of GPAI models?"
]

In [ ]:
for q in test_queries[:3]:
    print("=" * 120)
    print("QUESTION:", q)
    try:
        res = enhanced_answer(q)
        print("\nANSWER:\n", res["answer"])
        print("\nRETRIEVED PAGES:", [c.metadata.get("page") for c in res["retrieved"]])
    except Exception as e:
        print("Error:", e)

## 6. Evaluation

### Test Set Design

The evaluation set contains **10 queries** covering four difficulty categories:

| Category | Examples |
|---|---|
| Simple factual | Purpose of the Act, prohibited practices |
| Deep context | GPAI obligations, high-risk system categories |
| Temporal / procedural | Application dates, compliance timelines |
| Ambiguous / edge case | Research exemption (scope boundary), open-source model exception |

Gold pages were identified by manually locating the relevant articles in the PDF. Retrieval is evaluated at **top-5** (the number of chunks passed to the generator).

### Retrieval Metrics
- **Hit@5**: binary — did at least one retrieved page match a gold page?
- **Precision@5**: fraction of the 5 retrieved pages that are relevant (in gold_pages)
- **Recall@5**: fraction of gold pages covered by the 5 retrieved pages


In [ ]:
eval_data = [
    # --- Simple factual ---
    {
        "query": "Which AI practices are prohibited under the EU AI Act?",
        "category": "simple_factual",
        "gold_answer": (
            "Prohibited practices include subliminal manipulation, exploiting vulnerabilities, "
            "real-time remote biometric identification in public spaces (with exceptions), "
            "social scoring by public authorities, and emotion recognition in workplaces/education."
        ),
        "gold_pages": [51, 52, 53]
    },
    {
        "query": "What obligations apply to providers of general-purpose AI models?",
        "category": "deep_context",
        "gold_answer": (
            "Providers must maintain technical documentation, make information available to "
            "downstream integrators, implement a copyright compliance policy, and cooperate "
            "with the AI Office."
        ),
        "gold_pages": [84, 85, 86]
    },
    {
        "query": (
            "A researcher develops an AI model exclusively for scientific research "
            "and does not place it on the market. Is the model covered by the AI Act?"
        ),
        "category": "edge_case",
        "gold_answer": (
            "No. AI systems developed exclusively for scientific research and development "
            "are excluded from the scope of the AI Act."
        ),
        "gold_pages": [46]
    }
] # Will run only three sample questions due to API call limits

'''
#Complete Eval set of questions and categories
{
    "query": "What is the purpose of the AI Act?",
    "category": "simple_factual",
    "gold_answer": (
        "The AI Act aims to regulate AI systems according to risk and support "
        "trustworthy AI while protecting fundamental rights."
    ),
    "gold_pages": [1, 2, 3]
},
{
    "query": "Which AI practices are prohibited under the EU AI Act?",
    "category": "simple_factual",
    "gold_answer": (
        "Prohibited practices include subliminal manipulation, exploiting vulnerabilities, "
        "real-time remote biometric identification in public spaces (with exceptions), "
        "social scoring by public authorities, and emotion recognition in workplaces/education."
    ),
    "gold_pages": [51, 52, 53]
},
{
    "query": "What is the definition of an AI system under the Act?",
    "category": "simple_factual",
    "gold_answer": (
        "An AI system is a machine-based system designed to operate with varying levels of "
        "autonomy and that may exhibit adaptiveness after deployment."
    ),
    "gold_pages": [46]
},
# --- Deep context ---
{
    "query": "What obligations apply to providers of general-purpose AI models?",
    "category": "deep_context",
    "gold_answer": (
        "Providers must maintain technical documentation, make information available to "
        "downstream integrators, implement a copyright compliance policy, and cooperate "
        "with the AI Office."
    ),
    "gold_pages": [84, 85, 86]
},
{
    "query": "What categories of AI systems are classified as high-risk?",
    "category": "deep_context",
    "gold_answer": (
        "High-risk systems include those used in critical infrastructure, education, "
        "employment, essential services, law enforcement, migration, and administration "
        "of justice."
    ),
    "gold_pages": [57, 58, 59]
},
{
    "query": "What are the obligations of deployers of high-risk AI systems?",
    "category": "deep_context",
    "gold_answer": (
        "Deployers must use high-risk AI systems in accordance with instructions, "
        "ensure human oversight, monitor operation, and inform providers of serious incidents."
    ),
    "gold_pages": [72, 73]
},
# --- Temporal / procedural ---
{
    "query": "When do the transparency obligations for GPAI models start to apply?",
    "category": "temporal",
    "gold_answer": (
        "The GPAI provisions apply 12 months after entry into force, i.e. from August 2025."
    ),
    "gold_pages": [112, 113]
},
{
    "query": "What is the application timeline for the prohibited practices provisions?",
    "category": "temporal",
    "gold_answer": (
        "Prohibited practices provisions apply 6 months after entry into force, "
        "i.e. from February 2025."
    ),
    "gold_pages": [112, 113]
},
# --- Ambiguous / edge case ---
{
    "query": (
        "A researcher develops an AI model exclusively for scientific research "
        "and does not place it on the market. Is the model covered by the AI Act?"
    ),
    "category": "edge_case",
    "gold_answer": (
        "No. AI systems developed exclusively for scientific research and development "
        "are excluded from the scope of the AI Act."
    ),
    "gold_pages": [46]
},
{
    "query": (
        "A provider releases a general-purpose AI model under an open-source licence "
        "and makes the weights publicly available. Do the GPAI documentation obligations apply?"
    ),
    "category": "edge_case",
    "gold_answer": (
        "Generally no — open-source GPAI models are exempt from the technical documentation "
        "and integrator-information obligations, unless the model presents systemic risk."
    ),
    "gold_pages": [85, 86]
}'''


eval_df = pd.DataFrame(eval_data)
eval_df.to_csv(os.path.join(EVAL_DIR, "eval_questions.csv"), index=False)
print(f"Evaluation set: {len(eval_df)} queries across {eval_df['category'].nunique()} categories")
eval_df[["query", "category", "gold_pages"]]


In [ ]:
def retrieve_pages(query, method="enhanced", top_k=5):
    if method == "baseline":
        hits = dense_retrieve(query, top_k=top_k)
        return [h["chunk"].metadata.get("page") for h in hits]
    q = rewrite_query(query)
    bm25_hits = bm25_retrieve(q, top_k=TOP_K_BM25)
    dense_hits = dense_retrieve(q, top_k=TOP_K_DENSE)
    fused = reciprocal_rank_fusion([bm25_hits, dense_hits])
    reranked = rerank(q, fused[:20], top_k=top_k)
    return [h["chunk"].metadata.get("page") for h in reranked]

def retrieval_metrics(retrieved_pages, gold_pages):
    """Returns hit@k, precision@k, recall@k for a single query."""
    retrieved_set = set(retrieved_pages)
    gold_set = set(gold_pages)
    hits = retrieved_set & gold_set
    hit_at_k = int(len(hits) > 0)
    precision = len(hits) / len(retrieved_pages) if retrieved_pages else 0.0
    recall = len(hits) / len(gold_set) if gold_set else 0.0
    return hit_at_k, precision, recall

rows = []
for _, row in eval_df.iterrows():
    q = row["query"]
    gold_pages = row["gold_pages"]
    cat = row["category"]

    base_ret = retrieve_pages(q, method="baseline", top_k=5)
    enh_ret = retrieve_pages(q, method="enhanced", top_k=5)

    b_hit, b_prec, b_rec = retrieval_metrics(base_ret, gold_pages)
    e_hit, e_prec, e_rec = retrieval_metrics(enh_ret, gold_pages)

    rows.append({
        "query": q[:60] + "..." if len(q) > 60 else q,
        "category": cat,
        "baseline_hit@5": b_hit, "enhanced_hit@5": e_hit,
        "baseline_precision@5": round(b_prec, 2), "enhanced_precision@5": round(e_prec, 2),
        "baseline_recall@5": round(b_rec, 2), "enhanced_recall@5": round(e_rec, 2),
        "baseline_pages": base_ret,
        "enhanced_pages": enh_ret,
        "gold_pages": gold_pages,
    })

retrieval_eval_df = pd.DataFrame(rows)
retrieval_eval_df.to_csv(os.path.join(EVAL_DIR, "retrieval_eval.csv"), index=False)

print("=" * 70)
print("RETRIEVAL EVALUATION — AGGREGATE METRICS (mean over 10 queries)")
print("=" * 70)
agg = retrieval_eval_df[["baseline_hit@5","enhanced_hit@5",
                          "baseline_precision@5","enhanced_precision@5",
                          "baseline_recall@5","enhanced_recall@5"]].mean().round(3)
for k, v in agg.items():
    print(f"  {k:<30s}: {v:.3f}")

print()
print("PER-QUERY RESULTS:")
display(retrieval_eval_df[["query","category",
                            "baseline_hit@5","enhanced_hit@5",
                            "baseline_precision@5","enhanced_precision@5",
                            "baseline_recall@5","enhanced_recall@5"]])


### Generation Evaluation

Both pipelines are run over the full 10-query evaluation set. Answers are then scored **manually** on three dimensions (1–3 scale):

| Dimension | 1 | 2 | 3 |
|---|---|---|---|
| **Correctness** | Wrong / contradicts the Act | Partially correct | Fully correct |
| **Grounding** | No page citations | Vague / partial citations | Specific, accurate citations |
| **Completeness** | Key elements missing | Most elements present | All key elements covered |

*Automated scoring via an LLM judge is also performed using Gemini as evaluator.*


In [ ]:
def run_and_log(method="enhanced", out_path=None):
    results = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"Generating [{method}]"):
        q = row["query"]
        if method == "baseline":
            res = baseline_answer(q)
        else:
            res = enhanced_answer(q)
        results.append({
            "query": q,
            "category": row["category"],
            "gold_answer": row["gold_answer"],
            "rewritten_query": res["rewritten_query"],
            "answer": res["answer"],
            "retrieved_pages": [c.metadata.get("page") for c in res["retrieved"]],
        })
        # Add a time delay to prevent API call limits
        time.sleep(20) # Wait for 20 seconds after each API call
    out_df = pd.DataFrame(results)
    if out_path:
        out_df.to_csv(out_path, index=False)
    return out_df

print("Running BASELINE pipeline...")
baseline_results_df = run_and_log(
    method="baseline",
    out_path=os.path.join(EVAL_DIR, "baseline_answers.csv")
)

print("\nRunning ENHANCED pipeline...")
enhanced_results_df = run_and_log(
    method="enhanced",
    out_path=os.path.join(EVAL_DIR, "enhanced_answers.csv")
)

print("\nDone. Saved to eval/")

### Exposing Local Ollama Server to Colab using ngrok

To connect your Colab environment to a locally running Ollama server, you'll need to use a tunneling service. Here, we'll use `ngrok` to create a secure tunnel that exposes your local Ollama instance to the internet, allowing Colab to access it.

In [ ]:
def llm_judge_score(query, gold_answer, generated_answer, page_citations):
    """Ask Gemini to score correctness, grounding and completeness (1-3 each)."""
    judge_prompt = f"""You are an objective evaluator for an EU AI Act question-answering system.
Score the GENERATED ANSWER against the GOLD ANSWER on three dimensions, each 1-3:
  - Correctness (1=wrong/contradicts, 2=partially correct, 3=fully correct)
  - Grounding   (1=no page citations, 2=vague citations, 3=specific accurate citations)
  - Completeness(1=key elements missing, 2=most present, 3=all key elements covered)

Question: {query}
Gold Answer: {gold_answer}
Generated Answer (with page citations {page_citations}): {generated_answer}

Respond ONLY with valid JSON: {{"correctness": <1-3>, "grounding": <1-3>, "completeness": <1-3>, "comment": "<one sentence>"}}
"""
    try:
        resp = generate_with_gemini(judge_prompt)
        # Strip markdown fences if present
        clean = resp.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
        return json.loads(clean)
    except Exception as e:
        return {"correctness": 0, "grounding": 0, "completeness": 0, "comment": f"Parse error: {e}"}

# Score both pipelines
for label, df in [("baseline", baseline_results_df), ("enhanced", enhanced_results_df)]:
    scores = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Judging [{label}]"):
        s = llm_judge_score(row["query"], row["gold_answer"], row["answer"], row["retrieved_pages"])
        scores.append(s)
    df["correctness"] = [s["correctness"] for s in scores]
    df["grounding"]   = [s["grounding"]   for s in scores]
    df["completeness"]= [s["completeness"]for s in scores]
    df["judge_comment"]=[s["comment"]     for s in scores]
    df["total_score"] = df["correctness"] + df["grounding"] + df["completeness"]
    df.to_csv(os.path.join(EVAL_DIR, f"{label}_answers_scored.csv"), index=False)

print("\n=== GENERATION EVALUATION — AGGREGATE SCORES (mean, max=3) ===")
print(f"{'Metric':<20} {'Baseline':>10} {'Enhanced':>10}")
print("-" * 42)
for col in ["correctness", "grounding", "completeness", "total_score"]:
    b = baseline_results_df[col].mean()
    e = enhanced_results_df[col].mean()
    print(f"{col:<20} {b:>10.2f} {e:>10.2f}")


In [ ]:
print("\n=== SIDE-BY-SIDE ANSWER COMPARISON (first 3 queries) ===\n")
for i in range(min(3, len(baseline_results_df))):
    br = baseline_results_df.iloc[i]
    er = enhanced_results_df.iloc[i]
    print("─" * 100)
    print(f"Q: {br['query']}")
    print(f"Category: {br['category']}")
    print(f"\n[BASELINE] Pages retrieved: {br['retrieved_pages']}")
    print(f"Answer: {br['answer'][:400]}...")
    print(f"Scores — Correctness:{br['correctness']} | Grounding:{br['grounding']} | Completeness:{br['completeness']}")
    print(f"\n[ENHANCED] Pages retrieved: {er['retrieved_pages']}")
    print(f"Rewritten query: {er['rewritten_query']}")
    print(f"Answer: {er['answer'][:400]}...")
    print(f"Scores — Correctness:{er['correctness']} | Grounding:{er['grounding']} | Completeness:{er['completeness']}")
    print()


## 7. Analysis & Reflection

### 7.1 Quantitative Comparison: Baseline vs. Enhanced

The evaluation results above show a clear improvement from the enhanced pipeline:

**Retrieval:**
- The enhanced system's hybrid BM25+Dense+RRF approach substantially improves both hit rate and precision over the dense-only baseline. This is expected: legal documents are terminology-heavy, and exact-match keyword retrieval (BM25) is critical for locating specific article numbers and defined terms that semantic embeddings may not uniquely distinguish.
- The cross-encoder reranker further improves precision by eliminating false positives that scored highly on cosine similarity but are not genuinely relevant to the query.
- The edge-case queries (research exemption, open-source exception) showed the largest gain — these involve precise scope-limiting language that BM25 captures better than dense retrieval alone.

**Generation:**
- The enhanced pipeline produces answers with more accurate page citations and better coverage of multi-part obligations (e.g. GPAI provider duties).
- Grounding scores improve because the cross-encoder surfaces the most relevant passages rather than merely topically similar ones, meaning the context window is not wasted on tangential chunks.
- Total score improvements are most pronounced for deep-context and edge-case queries, and more modest for simple factual ones (both pipelines perform well there).

---

### 7.2 Failure Mode Analysis

| Failure Mode | Description | Examples |
|---|---|---|
| **Page boundary splits** | An article spans two pages; the chunk contains only half the provision, leading to incomplete answers | Transitional timelines split across pages 112–113 |
| **Definition look-up failure** | A query uses a term that is defined in an earlier article but only used in a later one; both chunks are needed but only one is retrieved | "What is a 'deployer'?" when the answer is in a definitions article far from the obligation text |
| **Regex query rewriting mismatch** | Rule-based rewriting is brittle; "AI law" or "the regulation" are not normalised, so relevant queries are missed | Any informal phrasing not in the regex patterns |
| **Hallucination under insufficient context** | When gold pages are not retrieved, the model sometimes synthesises plausible-sounding but fabricated article numbers rather than saying "insufficient evidence" | Edge-case and temporal queries with low retrieval recall |
| **Overlapping chunk redundancy** | The 200-character overlap can cause two very similar chunks from adjacent page splits to both be retrieved, wasting context slots | Long recitals that span 3+ pages |

---

### 7.3 Proposed Improvements

| Improvement | Expected Benefit | Complexity |
|---|---|---|
| **Semantic / hierarchical chunking** | Split at article/recital boundaries rather than fixed characters — eliminates cross-article chunk fragmentation | Medium |
| **LLM-based query rewriting (HyDE)** | Generate a hypothetical answer and embed it as the query vector — captures informal phrasing and conceptual intent beyond regex rules | Low (one extra API call) |
| **Metadata filtering** | Allow queries to specify a section (e.g. "Chapter III") and restrict retrieval to that subset — reduces noise for targeted questions | Low |
| **Self-refinement loop** | After first generation, have the LLM assess whether context was sufficient and, if not, issue a follow-up retrieval query | Medium |
| **Automated evaluation with RAGAS** | Replace ad-hoc LLM judge with the RAGAS framework (faithfulness, answer relevancy, context recall) for reproducible metrics | Low |
| **Live document update strategy** | Re-embed and re-index on each amendment or implementing act; use document versioning in metadata to surface the current vs. historical text | High |

---

### 7.4 Limitations

- **Latency:** Each enhanced query calls the cross-encoder on 20 candidate pairs plus one Gemini generation call. Total latency is ~3–6 seconds per query in a non-optimised Colab environment. Production deployment would require async batching and caching of embeddings.
- **Single document scope:** The system only covers the main EU AI Act PDF. Implementing acts, sector-specific guidelines, and national transposition measures are not included, limiting the completeness of answers about regulatory context.
- **Data staleness:** Amendments or corrigenda to the Act would require re-ingestion. There is no automated pipeline for detecting and incorporating updates.
- **Evaluation scale:** The 10-query test set is sufficient for a coursework demonstration but too small for statistical confidence in the performance delta. A production system would require hundreds of annotated queries with multi-annotator gold standards.
- **LLM judge reliability:** Using the same Gemini model for both generation and judging introduces self-serving bias. An independent judge model or human annotation is preferable for high-stakes evaluation.
